In [1]:
!pip install -q yt-dlp openai-whisper transformers sentencepiece
!apt-get install -y ffmpeg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.3/182.3 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 9.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 5.7 MB/s eta 0:00:00
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.


In [2]:
import whisper
import subprocess
from transformers import pipeline

In [3]:
youtube_url = "https://youtu.be/EDb37y_MhRw?si=GZyo3Np2v1TVpYth"

In [4]:
audio_file = "yt_audio.wav"

subprocess.run([
    "yt-dlp",
    "-x", "--audio-format", "wav",
    "-o", audio_file,
    youtube_url
])

print("Audio extracted from YouTube video")

Audio extracted from YouTube video


In [5]:
clean_audio = "audio_clean.wav"

subprocess.run([
    "ffmpeg", "-y",
    "-i", audio_file,
    "-ac", "1",
    "-ar", "16000",
    clean_audio
])

print("Audio re‑encoded for Whisper")

Audio re‑encoded for Whisper


In [6]:
whisper_model = whisper.load_model("base")

result = whisper_model.transcribe(clean_audio)
transcribed_text = result["text"]

print("\n📝 Transcribed Text:\n")
print(transcribed_text)

100%|████████████████████████████████████████| 139M/139M [00:01<00:00, 130MiB/s]
/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")



📝 Transcribed Text:

 What's the difference between generative AI and agentec AI? Well, they're two distinct approaches to artificial intelligence. And I think we're all familiar with generative AI, things like chatbots and image generators and the like. And they are really fundamentally reactive systems. They wait for you to do something. Specifically, they wait for you to prompt them. And once you prompt them, their job is to generate some kind of content based upon what you provided in the prompt. And they're using pansley learned during training. Now, the things that it can generate, well, that might be some text or it might be an image or it might be a piece of code or it might be some audio. These are all sorts of things that we can generate with generative AI. And they're essentially sophisticated pattern matching machines. They've learned the statistical relationships between words and between pixels and between sound waves. And they've learned that from massive data sets. So 

In [7]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("facebook/nllb-200-distilled-600M")
model = AutoModelForSeq2SeqLM.from_pretrained("facebook/nllb-200-distilled-600M")

tokenizer.src_lang = "eng_Latn"

inputs = tokenizer(
    transcribed_text,
    return_tensors="pt",
    truncation=True,
    max_length=512
)

# Hindi language code
forced_bos_token_id_hi = tokenizer.convert_tokens_to_ids("hin_Deva")

translated_tokens_hi = model.generate(
    **inputs,
    forced_bos_token_id=forced_bos_token_id_hi,
    max_length=600
)

hindi_full_translation = tokenizer.decode(
    translated_tokens_hi[0],
    skip_special_tokens=True
)

print("🇮🇳 Hindi Translation (Full Audio):\n")
print(hindi_full_translation)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

🇮🇳 Hindi Translation (Full Audio):

खैर, वे कृत्रिम बुद्धिमत्ता के लिए दो अलग-अलग दृष्टिकोण हैं। और मुझे लगता है कि हम सभी जनरेटिव एआई के बारे में जानते हैं, जैसे चैटबॉट और छवि जनरेटर आदि। और वे वास्तव में मौलिक रूप से प्रतिक्रियाशील एजेंट सिस्टम हैं। वे वास्तव में आपके लिए कुछ करने की प्रतीक्षा करते हैं। वे विशेष रूप से आपके द्वारा उन्हें प्रेरित करने की प्रतीक्षा करते हैं। और एक बार जब आप उन्हें प्रेरित करते हैं, तो उनका काम प्रॉम्प्ट बॉट में आपके द्वारा प्रदान किए गए सामग्री के आधार पर कुछ सामग्री उत्पन्न करना है। और वे प्रशिक्षण के दौरान पंसली का उपयोग कर रहे हैं। अब, वे जो चीजें उत्पन्न कर सकते हैं, वे कुछ पाठ साझा कर सकते हैं, ठीक है, या यह एक छवि हो सकती है या यह एक कोड का टुकड़ा हो सकता है। ये सभी प्रकार की चीजें हैं जो हम जनरेटिव एआईएम के साथ उत्पन्न कर सकते हैं। और इसलिए वे अनिवार्य रूप से परिष्कृत हैं। और इसलिए वे एक जटिल रूप से उत्पन्न होते हैं। और फिर वे एक बार जब वे एक बार इन सभी प्रकार के लिए काम करते हैं, तो वे एक सामान्य रूप से काम करते हैं, लेकिन अब आप एक बार एआईएम के

In [8]:
forced_bos_token_id_te = tokenizer.convert_tokens_to_ids("tel_Telu")

translated_tokens_te = model.generate(
    **inputs,
    forced_bos_token_id=forced_bos_token_id_te,
    max_length=600
)

telugu_full_translation = tokenizer.decode(
    translated_tokens_te[0],
    skip_special_tokens=True
)

print("🇮🇳 Telugu Translation (Full Audio):\n")
print(telugu_full_translation)

🇮🇳 Telugu Translation (Full Audio):

జనరేటివ్ AI మరియు ఏజెంట్ AI మధ్య తేడా ఏమిటి? బాగా, అవి కృత్రిమ మేధస్సు కోసం రెండు విభిన్న విధానాలను ఉపయోగిస్తాయి. శిక్షణ సమయంలో వారు నేర్చుకున్న పాన్స్లీని ఉపయోగిస్తున్నారు. ఇప్పుడు, ఇది ఉత్పత్తి చేయగల విషయాలు, చాట్బాట్లు మరియు ఇమేజ్ జెనరేటర్లు వంటివి మనకు బాగా తెలుసు. మరియు అవి నిజంగా ప్రాథమికంగా ప్రతిచర్య వ్యవస్థలు. అవి మీరు ఏదో చేయమని వేచి ఉంటాయి. ప్రత్యేకంగా, అవి మీరు వాటిని ప్రోత్సహించే వరకు వేచి ఉంటాయి. మరియు మీరు వాటిని ప్రోత్సహించిన తర్వాత, వారి పని మీరు ప్రోంట్ బాట్లలో అందించిన వాటి ఆధారంగా కొంత కంటెంట్ను ఉత్పత్తి చేయడం. మరియు వారు శిక్షణ సమయంలో నేర్చుకున్న ప్యాన్స్లీని ఉపయోగిస్తున్నారు. ఇప్పుడు, ఇది ఉత్పత్తి చేయగల విషయాలు, బాగా, ఇది కొంత టెక్స్ట్ను పంచుకోవచ్చు లేదా ఇది ఒక చిత్రం కావచ్చు లేదా ఇది ఒక కోడ్ ముక్క కావచ్చు లేదా ఇది కొన్ని ఆడియో ఏజెంట్లు కావచ్చు. ఇవి అన్ని రకాలైన విషయాలు మేము జనరేటివ్తో ఉత్పత్తి చేయగలము. మరియు ప్రత్యేకంగా, అవి చాలా సూక్ష్మంగా ఉత్పత్తి చేయగలవు. కాబట్టి అవి సాధారణంగా సూక్ష్మంగా ఉత్పత్తి చేయబడినవి మరియు అవి చాలా సూక

It's other part of transulation.[Other languages using choice]

In [9]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

In [10]:
model_name = "facebook/nllb-200-distilled-600M"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

tokenizer.src_lang = "eng_Latn"


Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [11]:
print("\nSelect a language to translate the audio text:\n")
print("1. Hindi")
print("2. Telugu")
print("3. Tamil")
print("4. French")
print("5. German")
print("6. Spanish")
print("7. Korean")

choice = input("\nEnter your choice (1–7): ")



Select a language to translate the audio text:

1. Hindi
2. Telugu
3. Tamil
4. French
5. German
6. Spanish
7. Korean

Enter your choice (1–7): 7


In [12]:
language_map = {
    "1": "hin_Deva",
    "2": "tel_Telu",
    "3": "tam_Taml",
    "4": "fra_Latn",
    "5": "deu_Latn",
    "6": "spa_Latn",
    "7": "kor_Hang"
}

if choice not in language_map:
    raise ValueError("Invalid choice selected")

target_language = language_map[choice]

In [13]:
inputs = tokenizer(
    transcribed_text,
    return_tensors="pt",
    truncation=True,
    max_length=512
)

forced_bos_token_id = tokenizer.convert_tokens_to_ids(target_language)

translated_tokens = model.generate(
    **inputs,
    forced_bos_token_id=forced_bos_token_id,
    max_length=600
)

translated_text = tokenizer.decode(
    translated_tokens[0],
    skip_special_tokens=True
)

print("\n🌍 Translated Text:\n")
print(translated_text)



🌍 Translated Text:

생성 AI와 에이전트 AI의 차이점은 무엇입니까? 음, 그들은 인공지능에 대한 두 가지 다른 접근 방식을 사용하고 있습니다. 그리고 우리는 모두 인공지능에 대해 알고 있습니다. 그리고 우리는 chatbot 및 이미지 생성기와 같은 것들을 생성 할 수 있습니다. 그리고 그들은 정말로 근본적으로 반응형 시스템입니다. 그들은 당신이 무언가를 수행하기를 기다리고 있습니다. 구체적으로, 그들은 당신이 그들을 촉구하는 것을 기다리고 있습니다. 그리고 일단 당신이 그들을 촉구하면, 그들의 일은 당신이 인스턴트 로봇에서 제공하는 것에 따라 어떤 종류의 콘텐츠를 생성하는 것입니다. 그리고 그들은 훈련 중에 배운 pansley를 사용합니다. 이제, 그것은 생성 할 수 있는 것, 물론, 그것은 텍스트 또는 이미지가 될 수 있습니다. 그것은 이미지가 될 수 있습니다 또는 그것은 일부 코드 조각이 될 수 있습니다. 이것은 우리가 생성하는 모든 종류의 것들입니다. 그리고 그들은 본질적으로 정리한 생성된 것입니다. 그래서 그들은 기본적으로 인식하는 모든 종류의 행동과 같은 복잡한 행동을 수행하는 것을 결정합니다. 그리고 그들은 종종 이러한 종류의 실행에 대한 행동과 실행에 대한 행동을 수행하는 것을 결정합니다. 그러나 이러한 일련의 실행에 의해 수행되는 모든 종류의 실행에 의해 실행되는 모든 종류의 행동과 실행에 대한 행동과 실행에 대한 행동을 결정합니다. 그러나 이제 우리는 이러한 일련의 실행에 의해 실행되는 모든 종류의 실행에 대한 행동을 수행 할 수 있습니다. 그러나 이러한 일련의 실행에 의해 실행된 일련의 실행에 의해 실행된 일련의 실행에 의해 실행된 일련의 실행에 의해 실행되는 모든 작업에 의해 실행되는 일련의 실행에 의해 실행되는 일련의 실행에 의해 실행되는 일련의 실행에 도달 할 수 있습니다.
